# Multi-zone CUDA fault — focused reproducer

On an A100 (torch 2.11 / CUDA 12.8) every multi-zone single-shooting case died with
`CUDA error: an illegal memory access was encountered` right after CUDA Graph capture, while 1-zone cases and
all laptop runs passed. The prime suspect is the benchmark's memory sampler thread, which polled
`torch.cuda.mem_get_info()` once a second: PyTorch captures graphs with `capture_error_mode="global"`, where a
CUDA runtime call from *any* thread invalidates the capture in flight. The 1-zone captures were short enough
to dodge a 1 Hz poll; the 10-zone captures were not. The fixed sampler only reads allocator bookkeeping.

This notebook runs **one** estimation case in-process (full traceback, no subprocess), with switches for the
things that discriminate the hypotheses:

* `MEMORY_SAMPLER` on/off — the old sampler is re-created here on demand so the fault can be reproduced deliberately;
* `BACKEND` `cuda_graph` vs `eager` — whether the fault needs graph capture at all;
* `CUDA_LAUNCH_BLOCKING` — makes an asynchronous CUDA error surface at the kernel that caused it;
* `MODE` `smoke` (2 h horizon, 1 iteration, minutes) vs `full` (120 h, 300 iterations).

Run the cells top to bottom. Cell 1 must run **before** anything imports torch, because `CUDA_LAUNCH_BLOCKING`
is read at CUDA initialisation.

In [ ]:
#@title 1. Configuration (runs before torch is imported) { display-mode: "form" }
import os
REPOSITORY = "https://github.com/JBjoernskov/Twin4Build.git"  #@param {type:"string"}
GIT_REF = "feature/issue-128/collocation-initialization"       #@param {type:"string"}
N_ZONES = 10                    #@param {type:"integer"}
SOLVER = "slsqp-single-shooting"  #@param ["slsqp-single-shooting", "custom-batched-sqp", "ipopt-collocation"]
N_STARTS = 1                    #@param {type:"integer"}
BACKEND = "cuda_graph"          #@param ["cuda_graph", "eager"]
MODE = "smoke"                  #@param ["smoke", "full"]
MEMORY_SAMPLER = "fixed"        #@param ["off", "fixed", "legacy_mem_get_info"]
CUDA_LAUNCH_BLOCKING = False    #@param {type:"boolean"}
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" if CUDA_LAUNCH_BLOCKING else "0"
os.environ["T4B_BENCHMARK_MODE"] = MODE
print({k: v for k, v in globals().items() if k in ("N_ZONES", "SOLVER", "N_STARTS", "BACKEND", "MODE", "MEMORY_SAMPLER", "CUDA_LAUNCH_BLOCKING")})

In [ ]:
#@title 2. Install Twin4Build from the branch (same bootstrap as the benchmark notebooks)
import pathlib, subprocess, sys
REPO = pathlib.Path("/content/Twin4Build")
if "google.colab" in sys.modules:
    if not REPO.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=False, capture_output=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "psutil"], check=True)
else:
    REPO = pathlib.Path.cwd()
    if REPO.name == "benchmarks":
        REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
import torch, twin4build
print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))

In [ ]:
#@title 3. Build the case exactly as the harness does
import time, json, traceback, threading
import benchmarks.common as common
from benchmarks.common import BenchmarkConfig, batched_estimation_problem, seed_everything, _estimation_window, ESTIMATION_METHODS
import twin4build as tb

config = BenchmarkConfig(mode=MODE)
seed_everything(config.seed)
t0 = time.time()
setup = batched_estimation_problem(N_ZONES, config)
model = setup["model"]; model.to("cuda", torch.float64)
print(f"setup built in {time.time()-t0:.0f} s: {N_ZONES} zones, hours={config.hours}, maxiter={config.estimation_maxiter}")

# Optional: re-create the OLD sampler (CUDA runtime call from a side thread) to reproduce deliberately.
legacy_stop = threading.Event()
def _legacy_poll():
    while not legacy_stop.wait(1.0):
        try:
            torch.cuda.mem_get_info()
        except Exception as exc:
            print("legacy sampler error:", repr(exc)[:200])
if MEMORY_SAMPLER == "legacy_mem_get_info":
    threading.Thread(target=_legacy_poll, daemon=True, name="legacy-mem-sampler").start()
    print("legacy mem_get_info sampler thread started (1 Hz)")
elif MEMORY_SAMPLER == "off":
    common.MEMORY_SAMPLER_INTERVAL_SECONDS = 3600.0  # effectively disables the fixed sampler

In [ ]:
#@title 4. Run the case in-process (full traceback on failure)
estimator = tb.Estimator(tb.Simulator(model, execution_mode="functional", execution_backend=BACKEND))
options = {"maxiter": config.estimation_maxiter}
method = ESTIMATION_METHODS[SOLVER]
if SOLVER == "ipopt-collocation":
    options["hessian"] = "exact"
if SOLVER == "custom-batched-sqp":
    options.update({"n_starts": N_STARTS, "batch_size": N_STARTS, "start_seed": config.seed, "start_strategy": "local", "start_spread": 0.1})
print("method", method, "options", options, "backend", BACKEND)
torch.cuda.reset_peak_memory_stats()
t0 = time.time(); outcome = None
try:
    result, seconds = common.timed("cuda", lambda: estimator.estimate(
        parameters=setup["parameters"], measurements=setup["measurements"], method=method, options=options, **_estimation_window(config)))
    outcome = {"status": "ok", "seconds": seconds, "iterations": result.get("iterations"), "message": result.get("message"),
               "final_objective": result.get("final_objective"), "derivative_stats": result.get("derivative_stats")}
except BaseException as exc:
    outcome = {"status": "FAILED", "seconds": time.time() - t0, "error": repr(exc)[:400]}
    traceback.print_exc()
finally:
    legacy_stop.set()
print(json.dumps(outcome, indent=1, default=str))
print("memory:", json.dumps({k: (round(v / 2**30, 2) if isinstance(v, int) and v > 1e6 else v) for k, v in common.memory_stats_of_last_timed().items()}, indent=1))
print("torch peak allocated GiB", round(torch.cuda.max_memory_allocated() / 2**30, 2), "reserved", round(torch.cuda.max_memory_reserved() / 2**30, 2))

### How to read the outcome

| Sampler | Backend | Expected if the sampler hypothesis is right |
|---|---|---|
| `legacy_mem_get_info` | `cuda_graph` | fails (illegal memory access, or capture invalidated) |
| `fixed` / `off` | `cuda_graph` | passes |
| any | `eager` | passes |

If `fixed` + `cuda_graph` still fails, re-run with `CUDA_LAUNCH_BLOCKING = True`: the traceback then names the
kernel, and `MODE = "smoke"` keeps the turnaround at minutes. Paste the traceback into the issue.

In [ ]:
#@title 5. (Optional) the same case through the real harness subprocess, with the fixed sampler
RUN_HARNESS = False  #@param {type:"boolean"}
if RUN_HARNESS:
    from benchmarks.common import _run_estimation_case_subprocess
    row = _run_estimation_case_subprocess(config, n_zones=N_ZONES, device="cuda", solver=SOLVER, n_starts=N_STARTS, repetition=0)
    print({k: row.get(k) for k in ("status", "seconds", "iterations", "message", "error", "child_returncode")})
    if row.get("child_traceback"):
        print(row["child_traceback"][-3000:])